In [2]:
# ── STEP 1: UPLOAD FILES ───────────────────────────────────
from google.colab import files
uploaded = files.upload()


Saving Chikago.csv to Chikago.csv
Saving Neworleans.csv to Neworleans.csv


In [4]:
# ── STEP 2: CHECK FILES ARE THERE ──────────────────────────
import os
print("Files in Colab:")
for f in os.listdir('/content'):
    print(" →", f)


Files in Colab:
 → .config
 → Chikago.csv
 → Neworleans.csv
 → sample_data


In [5]:
# ── STEP 3: LOAD DATA ────────────────────────────────────────
import pandas as pd
import numpy as np

chi = pd.read_csv('/content/Chikago.csv')
no  = pd.read_csv('/content/Neworleans.csv')

print("\n✅ Files loaded!")
print(f"Chicago rows    : {len(chi)}")
print(f"New Orleans rows: {len(no)}")
print(f"\nChicago columns:\n{list(chi.columns)}")


✅ Files loaded!
Chicago rows    : 8660
New Orleans rows: 7189

Chicago columns:
['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_q

In [6]:
# ── STEP 4: ADD CITY COLUMN ────────────────────────────────
chi['city'] = 'Chicago'
no['city']  = 'New Orleans'

df = pd.concat([chi, no], ignore_index=True)
print(f"\nCombined rows: {len(df)}")


Combined rows: 15849


In [7]:
# ── STEP 5: KEEP ONLY NEEDED COLUMNS ──────────────────────
wanted = ['id','name','host_id','host_name','neighbourhood_group',
          'neighbourhood','latitude','longitude','room_type',
          'price','minimum_nights','number_of_reviews','last_review',
          'reviews_per_month','availability_365','city']

cols = [c for c in wanted if c in df.columns]
df = df[cols]

if 'neighbourhood_group' not in df.columns:
    df['neighbourhood_group'] = 'Other'

print(f"\nColumns kept: {list(df.columns)}")



Columns kept: ['id', 'name', 'host_id', 'host_name', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'availability_365', 'city', 'neighbourhood_group']


In [8]:
# ── STEP 6: CLEAN PRICE COLUMN ─────────────────────────────
df['price'] = df['price'].astype(str).str.replace(r'[$,]', '', regex=True)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

df = df.dropna(subset=['price'])

df = df[(df['price'] > 0) & (df['price'] <= 2000)]
print(f"\nAfter price cleaning: {len(df)} rows")


After price cleaning: 13944 rows


In [9]:
# ── STEP 7: FILL MISSING VALUES ────────────────────────────
df['reviews_per_month']  = df['reviews_per_month'].fillna(0)
df['last_review']        = df['last_review'].fillna('Unknown')
df['neighbourhood_group']= df['neighbourhood_group'].fillna('Other')
df['neighbourhood']      = df['neighbourhood'].fillna('Unknown')


In [10]:
# ── STEP 8: REMOVE DUPLICATES ──────────────────────────────
before = len(df)
df = df.drop_duplicates(subset='id')
print(f"Duplicates removed: {before - len(df)}")

Duplicates removed: 0


In [11]:
# ── STEP 9: STANDARDISE NEIGHBOURHOOD NAMES ───────────────
df['neighbourhood'] = df['neighbourhood'].str.strip().str.title()

In [14]:
# ── STEP 10: CREATE EXTRA COLUMNS FOR TABLEAU ──────────────

# Price category
def price_bucket(p):
    if p < 50:    return 'Budget (< $50)'
    elif p < 100: return 'Economy ($50-100)'
    elif p < 150: return 'Mid-Range ($100-150)'
    elif p < 250: return 'Premium ($150-250)'
    else:         return 'Luxury ($250+)'

df['price_category'] = df['price'].apply(price_bucket)

# Availability category
def avail_bucket(a):
    if a <= 90:    return 'Low (0-90 days)'
    elif a <= 180: return 'Medium (91-180 days)'
    elif a <= 270: return 'High (181-270 days)'
    else:          return 'Always Available (271-365)'

df['availability_category'] = df['availability_365'].apply(avail_bucket)

# Review activity
def review_activity(r):
    if r == 0:    return 'No Reviews'
    elif r < 1:   return 'Occasional'
    elif r < 3:   return 'Regular'
    else:         return 'Highly Active'

df['review_activity'] = df['reviews_per_month'].apply(review_activity)

# Host type
host_counts = df.groupby('host_id')['id'].count().reset_index()
host_counts.columns = ['host_id', 'host_listing_count']
df = df.merge(host_counts, on='host_id', how='left')

def host_type(n):
    if n == 1:   return 'Casual (1 listing)'
    elif n <= 3: return 'Growing (2-3 listings)'
    else:        return 'Professional (4+ listings)'

df['host_type'] = df['host_listing_count'].apply(host_type)



In [15]:
# ── STEP 11: SAVE CLEAN FILE ───────────────────────────────
df.to_csv('/content/airbnb_clean.csv', index=False)
print("\n✅ Clean file saved as airbnb_clean.csv!")
print(f"Final rows : {len(df)}")
print(f"Final cols : {len(df.columns)}")
print(f"\nColumn list:\n{list(df.columns)}")
print(f"\nSample data:")
print(df[['city','neighbourhood','room_type','price','price_category','host_type']].head(5))



✅ Clean file saved as airbnb_clean.csv!
Final rows : 13944
Final cols : 23

Column list:
['id', 'name', 'host_id', 'host_name', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'availability_365', 'city', 'neighbourhood_group', 'price_category', 'availability_category', 'review_activity', 'host_listing_count_x', 'host_type', 'host_listing_count_y', 'host_listing_count']

Sample data:
      city neighbourhood        room_type   price        price_category  \
0  Chicago       Unknown     Private room  171.33    Premium ($150-250)   
1  Chicago       Unknown  Entire home/apt  111.50  Mid-Range ($100-150)   
2  Chicago       Unknown  Entire home/apt  280.16        Luxury ($250+)   
3  Chicago       Unknown     Private room  421.50        Luxury ($250+)   
4  Chicago       Unknown     Private room  108.67  Mid-Range ($100-150)   

                    host_type  
0          Casual (1 listing)  
1      

In [16]:
# ── STEP 12: DOWNLOAD THE CLEAN FILE ──────────────────────
from google.colab import files
files.download('/content/airbnb_clean.csv')
print("\n✅ Download started! Check your Downloads folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download started! Check your Downloads folder.
